# 🗂️ Notebook 2: Reminder / Alert — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/reminder-alert
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Entities

- **Reminder** — what, when, to whom, via what channel.
- **Delivery** — an attempt (for retry + audit).

## Pydantic models

We use `pydantic` for data validation — it forces us to think about types, required fields, and invariants up front.

In [ ]:
from datetime import datetime
from typing import Literal
from pydantic import BaseModel

class Reminder(BaseModel):
    id: int
    user_id: int
    fire_at: datetime   # UTC
    channel: Literal["push","email","sms"]
    payload: str
    status: Literal["scheduled","firing","done","cancelled"] = "scheduled"

class Delivery(BaseModel):
    reminder_id: int
    attempt: int
    status: Literal["ok","retry","dead"]
    ts: datetime

## HTTP APIs

| Method | Path | What |
|---|---|---|
| POST | `/reminders` | Schedule |
| DELETE | `/reminders/{id}` | Cancel |
| GET | `/reminders/{id}` | Status + delivery log |


## Quick demo

In [ ]:
# How the dispatcher selects ready reminders:
#   SELECT * FROM reminders
#   WHERE status='scheduled' AND fire_at <= now()
#   ORDER BY fire_at LIMIT 1000 FOR UPDATE SKIP LOCKED;
# SKIP LOCKED lets many workers share the queue without stepping on each other.
print("Pattern: index on (status, fire_at), then SELECT ... FOR UPDATE SKIP LOCKED")

## Takeaways

- Small, typed models make the service boundary crisp.
- Public APIs hide internal IDs and expose human-friendly resources.
- Write one happy-path test per endpoint before scaling out.